# Local Spiral Inductor Candidate-Space Run

This notebook defines a small Spiral Inductor candidate space, writes local queue jobs, optionally runs the local HFSS worker, and analyzes exported Touchstone files. HFSS execution stays behind the worker path; the notebook does not call PyAEDT directly.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / 'src').exists() and (path / 'configs').exists():
            return path
    raise RuntimeError('Could not locate repo root')


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
for path in (REPO_ROOT, SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from common.io_utils import read_json, write_json
from optimization.candidate_space import DiscreteCandidateSpace
from optimization.metrics import DifferentialInductanceAtFrequencyMetric, DifferentialQAtFrequencyMetric, DifferentialSRFMetric, SParameterAtFrequencyMetric, compute_sparameter_metric_values
from optimization.sparam_response import SParameterResponse
from scripts.worker.run_inductor_local_candidate_space import DEFAULT_INDUCTOR_RESOLUTION_UM, InductorDesignBounds, generate_lhs_geometries, normalize_inductor_resolution
from worker.param_strings import build_spiral_inductor_export_filename
from worker.schemas import JobRequest


: 

In [ ]:
# Configuration
N_CASES = 20
SEED = 0
H_UM = 2.0
RUN_WORKER = False

TEMPLATE_NAME = 'spiral_inductor_t3_template_v1'
DESIGN_NAME = 'SpiralInductorT3'
SETUP_NAME = 'Setup1'
SWEEP_NAME = 'Sweep'
METRICS_TO_EXTRACT = ['max_s11_db', 'min_s21_db']

EXPORT_DIR = Path('D:/ansys/spiral_inductor_s_parameters_export/ext')
STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ID = f'inductor_notebook_{STAMP}'
RUNTIME_ROOT = REPO_ROOT / '.runtime'
QUEUE_ROOT = RUNTIME_ROOT / 'queue'
LOGS_DIR = RUNTIME_ROOT / 'logs' / RUN_ID
TEMPLATE_COPY_ROOT = RUNTIME_ROOT / 'aedt_copies' / RUN_ID
GENERATED_WORKER_CONFIG = RUNTIME_ROOT / 'worker_configs' / f'{RUN_ID}.yaml'

BOUNDS = InductorDesignBounds(
    din_min_um=50.0,
    din_max_um=70.0,
    w_min_um=12.0,
    w_max_um=18.0,
    s_min_um=1.0,
    s_max_um=2.0,
    h_um=H_UM,
)
VARIABLE_ORDER = ['Din', 'W', 'S', 'H']
UNIT_MAP = {'Din': 'um', 'W': 'um', 'S': 'um', 'H': 'um'}
RESOLUTION_UM = normalize_inductor_resolution(DEFAULT_INDUCTOR_RESOLUTION_UM)

for path in [QUEUE_ROOT / 'pending', QUEUE_ROOT / 'running', QUEUE_ROOT / 'done', QUEUE_ROOT / 'failed', QUEUE_ROOT / 'results', EXPORT_DIR, LOGS_DIR, TEMPLATE_COPY_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print('run_id=', RUN_ID)
print('queue_root=', QUEUE_ROOT)
print('export_dir=', EXPORT_DIR)


In [ ]:
# Candidate space
geometries = generate_lhs_geometries(
    count=N_CASES,
    seed=SEED,
    bounds=BOUNDS,
    variable_order=VARIABLE_ORDER,
    resolution_um=RESOLUTION_UM,
)
candidate_space = DiscreteCandidateSpace(
    variable_order=VARIABLE_ORDER,
    candidate_geometries=geometries,
)

candidate_df = pd.DataFrame(geometries, columns=VARIABLE_ORDER)
candidate_df['geometry_valid'] = candidate_df['Din'] > 2.0 * (candidate_df['W'] + candidate_df['S'])
candidate_df['resolution_um'] = ';'.join(f'{name}=fixed' if step is None else f'{name}={step:g}um' for name, step in RESOLUTION_UM.items())
assert len(candidate_df) == N_CASES
assert candidate_df['geometry_valid'].all()
assert np.allclose(candidate_df['Din'] / RESOLUTION_UM['Din'], np.round(candidate_df['Din'] / RESOLUTION_UM['Din']))
assert np.allclose(candidate_df['W'] / RESOLUTION_UM['W'], np.round(candidate_df['W'] / RESOLUTION_UM['W']))
assert np.allclose(candidate_df['S'] / RESOLUTION_UM['S'], np.round(candidate_df['S'] / RESOLUTION_UM['S']))
candidate_df.head()


In [ ]:
# Generate worker config and pending job JSON files.
def format_um(value: float) -> str:
    return f'{float(value):.2f}um'


def output_filename(index: int, row: pd.Series) -> str:
    return build_spiral_inductor_export_filename(
        row['Din'],
        row['S'],
        row['W'],
        counter=index,
        counter_width=3,
        value_precision=2,
    )


base_worker_config = yaml.safe_load((REPO_ROOT / 'configs' / 'worker.yaml').read_text(encoding='utf-8'))
worker_config = dict(base_worker_config)
worker_config.update(
    {
        'worker_name': f'{base_worker_config.get("worker_name", "hfss_win01")}_notebook',
        'queue_root': str(QUEUE_ROOT),
        'results_dir': str(EXPORT_DIR),
        'logs_dir': str(LOGS_DIR),
        'copy_template_per_job': True,
        'template_copy_root': str(TEMPLATE_COPY_ROOT),
        'preserve_local_project_copy': False,
        'new_desktop': True,
        'close_desktop_after_job': True,
        'default_export_touchstone': True,
        'default_export_csv': False,
    }
)
GENERATED_WORKER_CONFIG.parent.mkdir(parents=True, exist_ok=True)
GENERATED_WORKER_CONFIG.write_text(yaml.safe_dump(worker_config, sort_keys=False), encoding='utf-8')

job_rows = []
for index, row in candidate_df.iterrows():
    job_id = f'{RUN_ID}_it{index:04d}'
    variables = {
        'Din': format_um(row['Din']),
        'W': format_um(row['W']),
        'S': format_um(row['S']),
        'H': format_um(row['H']),
    }
    filename = output_filename(index, row)
    payload = {
        'job_id': job_id,
        'template_name': TEMPLATE_NAME,
        'design_name': DESIGN_NAME,
        'setup_name': SETUP_NAME,
        'sweep_name': SWEEP_NAME,
        'variables': variables,
        'output_filename': filename,
        'metrics_to_extract': METRICS_TO_EXTRACT,
        'artifact_options': {'export_touchstone': True, 'export_csv': False},
        'submit_time': datetime.now(timezone.utc).isoformat(),
        'retry_count': 0,
    }
    JobRequest.model_validate(payload)
    pending_path = QUEUE_ROOT / 'pending' / f'{job_id}.json'
    write_json(pending_path, payload)
    job_rows.append({**row.to_dict(), 'job_id': job_id, 'output_filename': filename, 'pending_path': str(pending_path)})

jobs_df = pd.DataFrame(job_rows)
assert jobs_df['output_filename'].is_unique
jobs_df[['job_id', 'Din', 'S', 'W', 'H', 'resolution_um', 'output_filename']].head()


In [ ]:
# Optional: run the local worker. This starts real HFSS solves when RUN_WORKER=True.
if RUN_WORKER:
    command = [
        sys.executable,
        str(REPO_ROOT / 'scripts' / 'worker' / 'run_local_queue_worker.py'),
        '--worker-config',
        str(GENERATED_WORKER_CONFIG),
        '--queue-root',
        str(QUEUE_ROOT),
        '--max-jobs',
        str(N_CASES),
        '--no-wait',
    ]
    completed = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True, check=False)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    print('returncode=', completed.returncode)
else:
    print('RUN_WORKER=False; pending jobs are ready under', QUEUE_ROOT / 'pending')


In [ ]:
# Collect queue results and parse exported Touchstone files.
def result_payload(job_id: str):
    for state in ('done', 'failed'):
        path = QUEUE_ROOT / state / f'{job_id}.result.json'
        if path.exists():
            return state, path, read_json(path)
    return 'pending_or_missing', None, None


def resolve_artifact_path(artifact: str | None) -> Path | None:
    if not artifact:
        return None
    path = Path(artifact)
    return path if path.is_absolute() else QUEUE_ROOT / path


rows = []
for _, job in jobs_df.iterrows():
    state, result_path, payload = result_payload(job['job_id'])
    artifacts = payload.get('artifacts', {}) if isinstance(payload, dict) else {}
    metrics = payload.get('metrics', {}) if isinstance(payload, dict) else {}
    touchstone_path = resolve_artifact_path(artifacts.get('touchstone') if isinstance(artifacts, dict) else None)
    row = {
        **job.to_dict(),
        'status': state,
        'result_json': str(result_path) if result_path else None,
        'touchstone': str(touchstone_path) if touchstone_path else None,
        'error_message': payload.get('error_message') if isinstance(payload, dict) else None,
        'max_s11_db': metrics.get('max_s11_db') if isinstance(metrics, dict) else None,
        'min_s21_db': metrics.get('min_s21_db') if isinstance(metrics, dict) else None,
        'L_5GHz_nH': np.nan,
        'Q_5GHz': np.nan,
        'SRF_GHz': np.nan,
        'S11_5GHz_dB': np.nan,
        'S21_5GHz_dB': np.nan,
    }
    if state == 'done' and touchstone_path is not None and touchstone_path.exists():
        import skrf as rf
        network = rf.Network(str(touchstone_path))
        target_hz = 5.0e9
        response = SParameterResponse.from_sparams(network.f, network.s)
        extracted = compute_sparameter_metric_values(
            response,
            [
                DifferentialInductanceAtFrequencyMetric(target_hz),
                DifferentialQAtFrequencyMetric(target_hz),
                DifferentialSRFMetric(),
                SParameterAtFrequencyMetric('S11', target_hz),
                SParameterAtFrequencyMetric('S21', target_hz),
            ],
        )
        row.update(extracted)
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df['target_L_error_nH'] = (summary_df['L_5GHz_nH'] - 1.2).abs()
done_mask = summary_df['status'].eq('done') & summary_df['target_L_error_nH'].notna() & summary_df['Q_5GHz'].notna()
summary_df.loc[done_mask, 'target_rank'] = (
    summary_df.loc[done_mask]
    .sort_values(['target_L_error_nH', 'Q_5GHz'], ascending=[True, False])
    .reset_index()
    .reset_index()
    .set_index('index')['level_0']
    + 1
)
summary_path = QUEUE_ROOT / 'results' / f'{RUN_ID}_inductor_notebook_summary.csv'
summary_df.to_csv(summary_path, index=False)
print('summary=', summary_path)
summary_df[['job_id', 'status', 'Din', 'S', 'W', 'H', 'L_5GHz_nH', 'Q_5GHz', 'SRF_GHz', 'touchstone', 'error_message']].head()


In [ ]:
# Parameter coverage.
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].scatter(summary_df['Din'], summary_df['W'], c=summary_df['S'], cmap='viridis')
axes[0].set_xlabel('Din (um)')
axes[0].set_ylabel('W (um)')
axes[0].set_title('Din-W')
axes[1].scatter(summary_df['Din'], summary_df['S'], c=summary_df['W'], cmap='plasma')
axes[1].set_xlabel('Din (um)')
axes[1].set_ylabel('S (um)')
axes[1].set_title('Din-S')
axes[2].scatter(summary_df['W'], summary_df['S'], c=summary_df['Din'], cmap='cividis')
axes[2].set_xlabel('W (um)')
axes[2].set_ylabel('S (um)')
axes[2].set_title('W-S')
fig.tight_layout()


In [ ]:
# Response metrics and top candidates.
done_df = summary_df[summary_df['status'] == 'done'].copy()
if not done_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    for ax, col in zip(axes, ['L_5GHz_nH', 'Q_5GHz', 'SRF_GHz']):
        ax.hist(done_df[col].dropna(), bins=min(10, max(3, len(done_df))))
        ax.set_title(col)
    fig.tight_layout()

    fig, ax = plt.subplots(figsize=(6, 4))
    scatter = ax.scatter(done_df['L_5GHz_nH'], done_df['Q_5GHz'], c=done_df['SRF_GHz'], cmap='viridis')
    ax.axvline(1.2, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('L at 5 GHz (nH)')
    ax.set_ylabel('Q at 5 GHz')
    ax.set_title('L-Q')
    fig.colorbar(scatter, ax=ax, label='SRF (GHz)')
    fig.tight_layout()

top_candidates = summary_df.sort_values(['target_rank'], na_position='last').head(10)
top_candidates[['target_rank', 'job_id', 'Din', 'S', 'W', 'H', 'L_5GHz_nH', 'Q_5GHz', 'SRF_GHz', 'touchstone']]
